In [1]:
using LatticeDiracOperators
using LatticeQCD
using Gaugefields
using Wilsonloop
using LMP
using LinearAlgebra
using JLD2

[ Info: Precompiling LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018]
[ Info: Skipping precompilation since __precompile__(false). Importing LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018].
┌ Warning: Error requiring `Plots` from `LatticeQCD`
│   exception =
│    LoadError: ArgumentError: Package LatticeQCD does not have Plots in its dependencies:
│    - You may have a partially installed environment. Try `Pkg.instantiate()`
│      to ensure all packages in the environment are installed.
│    - Or, if you have LatticeQCD checked out for development and have
│      added Plots as a dependency but haven't updated your primary
│      environment's manifest file, try `Pkg.resolve()`.
│    - Otherwise you may need to report an issue with LatticeQCD
│    Stacktrace:
│      [1] macro expansion
│        @ .\loading.jl:1839 [inlined]
│      [2] macro expansion
│        @ .\lock.jl:267 [inlined]
│      [3] __require(into::Module, mod::Symbol)
│        @ Base .\loading.jl:1816
│      [4] #invoke_in_world#3


In [2]:
NX = 1
NY = 1
NZ = 1
NT = 1
Nwing = 0
Dim = 4
NC = 3
U = Initialize_4DGaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "hot")
x = Initialize_pseudofermion_fields(U[1],"Wilson",nowing=true)
gauss_distribution_fermion!(x)
t = Initialize_pseudofermion_fields(U[1],"Wilson",nowing=true)
gauss_distribution_fermion!(t)
x=x/sqrt(norm2(x))
t=t/sqrt(norm2(t))

LatticeDiracOperators.Dirac_operators.WilsonFermion_4D_nowing{3}(ComplexF64[0.19152505582590523 - 0.08531063300551875im; 0.16291242093102215 - 0.46096724025210883im; -0.24007180494581756 + 0.08736498645887583im;;;;;; -0.28664125425330134 - 0.051011310487975685im; -0.04933605482409528 + 0.2688917794822285im; -0.22727428000463024 + 0.040432083433651164im;;;;;; -0.2372459257980776 - 0.016339592836723583im; 0.018335408826931618 - 0.12982916065555966im; 0.33563795618324815 + 0.008786660737125905im;;;;;; 0.24748457569174268 - 0.3101531964274728im; -0.1063919362153437 + 0.054297359782211986im; -0.26832873886440145 - 0.09362425292703354im], 3, 1, 1, 1, 1, 4, 0, "Wilson", ComplexF64[0.0 + 0.0im; 0.0 + 0.0im; 0.0 + 0.0im;;;;;; 0.0 + 0.0im; 0.0 + 0.0im; 0.0 + 0.0im;;;;;; 0.0 + 0.0im; 0.0 + 0.0im; 0.0 + 0.0im;;;;;; 0.0 + 0.0im; 0.0 + 0.0im; 0.0 + 0.0im])

In [3]:
P=Onestep()
hop=Hopping_operator(U,P)
KW=Generate_weight(hop)
QW=Generate_weight(hop)
VW=Generate_weight(hop)
act=Softmax()
m=sqrt(length(x))

3.4641016151377544

In [4]:
function expo(x::Spin_matrix)
    expA = exp.(x.sm)
    result=Spin_matrix(expA)
    for a in 1:4
        for b in 1:4
            for c in 1:4
                result[a,b]+= result[a,b]/result[a,c]
            end
        end
    end
    return result
end

function expo(x::Vector{Spin_matrix})
    y = deepcopy(x)

    for i in 1:length(x)
        expA = expo(x[i])
        y[i] = Spin_matrix(expA )
    end

    return y
end

expo (generic function with 2 methods)

In [5]:
epsilon = 1e-8*im
grad = similar(x)  # 이미 0 초기화가 보장된 상태라고 가정

# 기준값 계산
Kx = Clifford(KW)*hop(x)
Qx = Clifford(QW)*hop(x)
Vx = Clifford(VW)*hop(x)

M = similar(KW)
S = similar(KW)

for i in 1:length(KW)
    M[i] = Matmul!(conj(Qx[i]), Kx[i])/m
    S[i] = act(M[i])
    # S[i]=M[i]
end




y = similar(t)  # similar가 0 초기화라고 가정
for i in 1:length(KW)
    y+=S[i]*Vx[i]
end


f = norm2(y-t)

# 수치 미분
for ic = 1:x.NC, it = 1:x.NT, iz = 1:x.NZ, iy = 1:x.NY, ix = 1:x.NX, fg = 1:x.NG

    d = deepcopy(x)
    d[ic, ix, iy, iz, it, fg] += epsilon

    Kd = Clifford(KW)*hop(d)
    Qd = Clifford(QW)*hop(d)
    Vd = Clifford(VW)*hop(d)

    dM = similar(KW)
    dS = similar(KW)

    for i in 1:length(KW)
        dM[i] = Matmul!(conj(Qd[i]), Kd[i])/m
        dS[i] = act(dM[i])
        # dS[i] = dM[i]
    end



    k = similar(t)  # similar가 0 초기화라고 가정
    for i in 1:length(KW)
        k += dS[i] * Vd[i]
    end




    grad[ic, ix, iy, iz, it, fg] = (norm2(k-t) - f) / epsilon
end



In [6]:
delta = similar(x)  # 이미 0 초기화

# 정확한 gradient 방향 (손실이 norm^2이므로)
d = similar(x)

# eta 정확한 계산 (필수!)
eta = similar(KW)
bareta = similar(KW)

for i in 1:length(KW)
    eta[i] = Matmul!(Vx[i], conj(y-t))

end

for i in 1:length(KW)
    
    bareta[i]= Matmul!(conj(Vx[i]), y-t)

end
# bareta= Matmul!(y, conj(t))


# Kz 계산 (정확한 버전)
Kz1 = similar(KW)

for i in 1:length(KW)
    for a in 1:4, c in 1:4 , m in 1:4
        Kz1[i][a,c] +=Clifford(KW)[i][m,a]* S[i][c,m]*eta[i][m,c]
    end
end

Kz2 = similar(KW)

for i in 1:length(KW)
    for a in 1:4, d in 1:4, c in 1:4 , m in 1:4
        Kz2[i][a,c] +=Clifford(KW)[i][d,a]*S[i][c,m]*S[i][c,d]*eta[i][m,c]
    end
end
Kz = Kz1 - Kz2




# # 최종 delta 계산
for i in 1:length(KW)
    delta += Kz[i]*conj(Clifford(QW)[i]*x)/m
end




Qz1 = similar(QW)
Qz2 = similar(QW)


for i in 1:length(QW)
    for a in 1:4,  c in 1:4, d in 1:4
        Qz1[i][a,c] +=Clifford(QW)[i][d,a]*(adjoint(S[i])[c,d])*bareta[i][c,d]
    end
end

for i in 1:length(QW)
    for a in 1:4, c in 1:4, e in 1:4 , d in 1:4
        Qz2[i][a,c] +=  Clifford(QW)[i][d,a]*(adjoint(S[i])[e,d])*(adjoint(S[i])[c,d]) * bareta[i][e,d]
    end
end







Qz=Qz1-Qz2

for i in 1:length(QW)
    delta+=Qz[i]*conj(Clifford(KW)[i]*x)/m

end




invP = reverse_paths(P)
invhop = Hopping_operator(U, invP)

Vz=similar(VW)

for i in 1:length(VW)
    for a in 1:4
        for c in 1:4
            for e in 1:4
                Vz[i][a,c]+=Clifford(VW)[i][e,a]*S[i][c,e]
            end
        end
    end
end


dH=invhop(y-t)

for i in 1:length(VW)
    delta+=Vz[i]*conj(dH[i])
end    



In [7]:
delta

LatticeDiracOperators.Dirac_operators.WilsonFermion_4D_nowing{3}(ComplexF64[33.828414503278154 - 9.21155386604817im; 26.57097516006419 + 8.75066055942935im; 1.017725477956406 - 26.47929678935929im;;;;;; 1.4121955080734132 + 7.297611613996622im; -53.71778614428283 + 0.13280172705374227im; 47.78360273709951 + 9.473920287937068im;;;;;; 24.391082565805217 - 38.84793290082544im; 14.834130146387324 + 3.915113230271822im; -29.79157264122514 - 7.49608500881655im;;;;;; -31.267704807157973 + 24.490700438358292im; -27.62962972306605 - 4.077918848794296im; 45.22083472881361 + 13.164410416108046im], 3, 1, 1, 1, 1, 4, 0, "Wilson", ComplexF64[-3.454492576592938 - 1.10043979624829im; 2.7093526840903683 + 0.3069787671475313im; 0.08254048229619557 + 0.6129166550609507im;;;;;; -2.2360771440865426 - 1.0113484471149274im; 0.9209167165673271 + 0.007139048855236918im; -1.1984507014035897 + 1.932649811205055im;;;;;; -0.10259203790780866 - 1.545057568585803im; 1.9372556852995206 + 0.16422074310559942im; 0.0639

In [8]:
grad

LatticeDiracOperators.Dirac_operators.WilsonFermion_4D_nowing{3}(ComplexF64[0.0 - 18.423109793275216im; 0.0 + 17.501319860002695im; 0.0 - 52.95859750731324im;;;;;; 0.0 + 14.595224939739637im; 0.0 + 0.2656037167980685im; 0.0 + 18.9478406298349im;;;;;; 0.0 - 77.69586716221966im; 0.0 + 7.830222159554978im; 0.0 - 14.992169639072017im;;;;;; 0.0 + 48.9813984927423im; 0.0 - 8.155838315815345im; 0.0 + 26.328815749820933im], 3, 1, 1, 1, 1, 4, 0, "Wilson", ComplexF64[0.0 + 0.0im; 0.0 + 0.0im; 0.0 + 0.0im;;;;;; 0.0 + 0.0im; 0.0 + 0.0im; 0.0 + 0.0im;;;;;; 0.0 + 0.0im; 0.0 + 0.0im; 0.0 + 0.0im;;;;;; 0.0 + 0.0im; 0.0 + 0.0im; 0.0 + 0.0im])

In [15]:
2*delta-grad

LatticeDiracOperators.Dirac_operators.WilsonFermion_4D_nowing{3}(ComplexF64[67.65682900655631 + 2.061178875578662e-6im; 53.14195032012838 + 1.258856006103315e-6im; 2.035450955912812 + 3.928594665580931e-6im;;;;;; 2.8243910161468264 - 1.7117463926297205e-6im; -107.43557228856567 - 2.6269058395200773e-7im; 95.56720547419901 - 5.396076474539768e-8im;;;;;; 48.782165131610434 + 1.360568774089188e-6im; 29.668260292774647 + 4.3009886656619756e-6im; -59.58314528245028 - 3.7856108292544377e-7im;;;;;; -62.535409614315945 + 2.3839742837594713e-6im; -55.2592594461321 + 6.182267533461072e-7im; 90.44166945762721 + 5.082395158950703e-6im], 3, 1, 1, 1, 1, 4, 0, "Wilson", ComplexF64[0.0 + 0.0im; 0.0 + 0.0im; 0.0 + 0.0im;;;;;; 0.0 + 0.0im; 0.0 + 0.0im; 0.0 + 0.0im;;;;;; 0.0 + 0.0im; 0.0 + 0.0im; 0.0 + 0.0im;;;;;; 0.0 + 0.0im; 0.0 + 0.0im; 0.0 + 0.0im])

In [10]:
layer=CPTC_Layer(hop,QW)
layer1=CPTC_Layer(hop,QW)
layer1=test_Layer(layer1,sqrt(length(x)),t ,act )
layer1.Q.W=QW
layer1.K.W=KW
layer1.V.W=VW
ly=norm2(layer1(x)-t)

55.348776668562 + 0.0im

In [11]:
norm2(y-t)

55.34877666856201 + 0.0im

In [12]:
γ=Clifford()
u = forward_propagation([layer1], x)
fdelta = backward_propagation([layer1], t, u)
DeltaRule(layer1, layer1.activation, x, fdelta[1])

LatticeDiracOperators.Dirac_operators.WilsonFermion_4D_nowing{3}(ComplexF64[67.65682900655631 - 18.423107732096327im; 53.14195032012834 + 17.50132111885869im; 2.035450955912852 - 52.95859357871857im;;;;;; 2.8243910161468424 + 14.595223227993236im; -107.43557228856557 + 0.2656034541075254im; 95.56720547419899 + 18.94784057587414im;;;;;; 48.78216513161045 - 77.69586580165085im; 29.66826029277462 + 7.830226460543651im; -59.58314528245027 - 14.992170017633113im;;;;;; -62.53540961431593 + 48.98140087671658im; -55.25925944613207 - 8.155837697588595im; 90.44166945762719 + 26.32882083221609im], 3, 1, 1, 1, 1, 4, 0, "Wilson", ComplexF64[-6.908985153185876 - 2.2008795924965794im; 5.418705368180736 + 0.6139575342950631im; 0.16508096459239002 + 1.225833310121901im;;;;;; -4.472154288173085 - 2.0226968942298558im; 1.8418334331346546 + 0.01427809771047428im; -2.3969014028071793 + 3.8652996224101113im;;;;;; -0.2051840758156181 - 3.090115137171606im; 3.874511370599042 + 0.3284414862111985im; 0.12788096